# 1 · Distributed average consensus

Companion to **Section 3**. Consensus is the master problem with $\mathbf b=\mathbf 0$
and no constraint: $\dot{\mathbf x}=-L\mathbf x$ in continuous time,
$\mathbf x(k+1)=(I-\theta L)\mathbf x(k)$ in discrete time.

In [ ]:
import sys
sys.path.insert(0, "../code")          # the scripts that generate the paper's figures
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
np.set_printoptions(precision=4, suppress=True)


def draw(ax, P, edges, nodes_kw=None, title=None):
    """Draw a graph with node positions P (n x 2) and an edge list."""
    ax.add_collection(LineCollection([[P[i], P[j]] for i, j in edges],
                                     colors="0.75", linewidths=0.6, zorder=1))
    ax.scatter(P[:, 0], P[:, 1], s=14, zorder=2, **(nodes_kw or {}))
    ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)

In [ ]:
from common import graph_consensus, graph_ahp, spectra, SEED
G, A, d, L, P = graph_consensus()
lam, Q, *_ = spectra(L, d)
n, edges = len(d), list(G.edges())
print(f"n = {n}, m = {len(edges)}, d_max = {d.max():.0f}, "
      f"lambda_2 = {lam[1]:.4f}, lambda_n = {lam[-1]:.3f}, kappa = {lam[-1]/lam[1]:.1f}")
fig, ax = plt.subplots(figsize=(4, 4)); draw(ax, P, edges, title="sensor network"); plt.show()

## Continuous time

Since $L=Q\Lambda Q^\top$, the solution is $\mathbf x(t)=Q\,e^{-\Lambda t}Q^\top\mathbf x(0)$,
and the disagreement is bounded by $e^{-\lambda_2 t}\|\mathbf x(0)-x_{\rm ave}\mathbf 1\|$
(Corollary 3.1).

In [ ]:
rng = np.random.default_rng(SEED)
x0 = rng.uniform(-1, 1, n)
t = np.linspace(0, 60, 601)
X = Q @ (np.exp(-np.outer(lam, t)) * (Q.T @ x0)[:, None])
err = np.linalg.norm(X - x0.mean(), axis=0)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(t, X.T, lw=0.6); ax[0].set_xlim(0, 40); ax[0].set_xlabel("t"); ax[0].set_title("x_i(t)")
ax[1].semilogy(t, err, label="disagreement")
ax[1].semilogy(t, err[0] * np.exp(-lam[1] * t), "--", label="exp(-lambda_2 t)")
ax[1].set_xlabel("t"); ax[1].legend(); plt.show()
print("time for three digits, at most:", np.log(1e3) / lam[1])

## Discrete time and the step size

The rate is $\rho(\theta)=\max\{|1-\theta\lambda_2|,|1-\theta\lambda_n|\}$ and the
iteration converges iff $0<\theta<2/\lambda_n$ (Theorem 2.5).

In [ ]:
def run(theta, K=600):
    x, out = x0.copy(), []
    for _ in range(K):
        out.append(np.linalg.norm(x - x0.mean()))
        x = x - theta * (L @ x)
    return np.array(out)

rho = lambda th: max(abs(1 - th * lam[1]), abs(1 - th * lam[-1]))
steps = {"1/d_max": 1 / d.max(),
         "theta* = 2/(l2+ln)": 2 / (lam[1] + lam[-1]),
         "2.02/l_n (too large)": 2.02 / lam[-1]}
fig, ax = plt.subplots(figsize=(5, 3.5))
for name, th in steps.items():
    e = run(th)
    ax.semilogy(e, label=f"{name}: rho = {rho(th):.4f}")
    if rho(th) < 1:
        print(f"{name:22s} iterations for 1e-3: {np.ceil(np.log(1e3) / -np.log(rho(th))):.0f}")
ax.set_ylim(1e-6, 1e3); ax.set_xlabel("k"); ax.legend(fontsize=8); plt.show()

*Try it.* Replace `graph_consensus()` by `graph_ahp()` (the small world of
Section 6): same number of nodes, three times the algebraic connectivity.
How do the settling times change? Exercises 2 and 3 work out the optimal
step and a graph on which $\theta=1/d_{\max}$ fails.